In [1]:
import xarray as xr
import pandas as pd
from scipy.stats import linregress
import matplotlib.pyplot as plt

In [2]:
def classify_precip(row):
    code = row['WEATHER']
    date = row['DATE']
    year = date.year  # extract year from datetime

    if code == 5:
        return 'Non-Precipitation'
    elif code in [6, 7]:
        return 'Non-Precipitation' if year >= 2007 else 'Precipitation'
    else:
        return 'Precipitation'

In [3]:
def detrend_state(df):
    df = df.sort_values('DATE')
    x = df['DATE'].map(pd.Timestamp.toordinal)
    y = df['CRASH_COUNTS']
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    df['DETRENDED_CRASH_COUNTS'] = y - (slope*x + intercept)
    return df


In [4]:
def read_psl_standard(filename, missing_value_marker=-999):
    """
    Read a NOAA PSL "standard format" time-series file into a pandas DataFrame.

    This function parses the PSL standard format:
      - First line contains start and end years
      - Each subsequent line has a year followed by 12 monthly values
      - A sentinel line with a missing-value marker (e.g., -999) follows the last year
      - Any additional lines (metadata/info) are ignored

    Returns
    -------
    df : pandas.DataFrame
        Wide format with columns: ['year', 'month_1', ..., 'month_12'].
    df_long : pandas.DataFrame
        Long format with columns: ['year', 'month', 'value'].

    Note
    ----
    This code was generated with the assistance of ChatGPT.
    """

    with open(filename, 'r') as f:
        # First line: contains the start and end year (not used directly, but can be checked)
        first_line = f.readline().strip().split()
        year_start, year_end = map(int, first_line)

        data_lines = []
        while True:
            line = f.readline()
            if not line:
                # Safety: file ended before sentinel found
                raise ValueError("Reached EOF before missing-value marker")
            parts = line.strip().split()
            # Stop when we reach the sentinel missing-value marker (e.g., -999)
            if len(parts) == 1 and parts[0] == str(missing_value_marker):
                break
            data_lines.append(parts)

        # Create DataFrame with year + 12 monthly values
        df = pd.DataFrame(data_lines, columns=['YEAR'] + [f'MONTH_{i}' for i in range(1, 13)])
        # Convert year to int and monthly values to float
        df = df.astype(float)
        df['YEAR'] = df['YEAR'].astype(int)

        # Reshape into long format for easier time-series analysis
        df_long = df.melt(id_vars='YEAR', var_name='MONTH', value_name='SST_VALUE')
        # Convert 'month' from "month_1" → integer 1
        df_long['MONTH'] = df_long['MONTH'].str.replace('MONTH_', '').astype(int)

        return df, df_long

In [5]:
state_dictAB = {
    1: 'AL', 2: 'AK', 4: 'AZ', 5: 'AR', 6: 'CA', 8: 'CO', 9: 'CT', 10: 'DE',
    11: 'DC', 12: 'FL', 13: 'GA', 15: 'HI', 16: 'ID', 17: 'IL', 18: 'IN',
    19: 'IA', 20: 'KS', 21: 'KY', 22: 'LA', 23: 'ME', 24: 'MD', 25: 'MA',
    26: 'MI', 27: 'MN', 28: 'MS', 29: 'MO', 30: 'MT', 31: 'NE', 32: 'NV',
    33: 'NH', 34: 'NJ', 35: 'NM', 36: 'NY', 37: 'NC', 38: 'ND', 39: 'OH',
    40: 'OK', 41: 'OR', 42: 'PA', 44: 'RI', 45: 'SC', 46: 'SD', 47: 'TN',
    48: 'TX', 49: 'UT', 50: 'VT', 51: 'VA', 53: 'WA', 54: 'WV', 55: 'WI',
    56: 'WY'
}

In [6]:
excluded_states = {'DC', 'AK', 'HI', 'PR', 'PRCP'}

In [7]:
cluster_names = {
    0: "Alaskan Ridge/Pacific Ridge",
    1: "Greenland High",
    2: "Pacific Trough",
    3: "West Coast High"
}

# Files to Read

In [8]:
# FARS
fars_file='../data/FARS/old/DATABASE3.csv'

# Weather Regimes Precip by State
wx_regimes_file='../data/precip/state_precip_wxregimes.csv'

# Weather Regime designation by date
cluster_file='/data/esplab/scratch/kpegion/obs_seus/wr_z500/kmeans_4cluster_DJF_1981-2019_NA.nc'

# State Precip from CHIRPS
precip_file='../data/precip/state_precip_chirps.csv'

# 1. FARS DATA -> Daily

In [9]:
# Read Data
df_fars=pd.read_csv(fars_file)

# map states
df_fars['STATE_ABBR'] = df_fars['STATE'].map(state_dictAB)

# remove excluded states
df_fars=df_fars[~df_fars["STATE_ABBR"].isin(excluded_states)]

# Subset Years in fars
df_fars = df_fars[(df_fars["YEAR"] >= 1981) & (df_fars["YEAR"] <= 2019)]

# Build datetime column in df_fars
df_fars["DATE"] = pd.to_datetime(
    df_fars["YEAR"].astype(str).str.zfill(4)
    + df_fars["MONTH"].astype(str).str.zfill(2)
    + df_fars["DAY"].astype(str).str.zfill(2),
    format="%Y%m%d",
    errors="coerce"  # handles bad dates automatically
)

# Ensure DATE is datetime
df_fars["DATE"] = pd.to_datetime(df_fars["DATE"])

# Keep only precipitation-classified crash events
df_fars["precip_class"] = df_fars.apply(classify_precip, axis=1)
df_fars_precip = df_fars[df_fars["precip_class"] == "Precipitation"].copy()
df_fars_precip.drop(columns=["precip_class"], inplace=True)

# Daily aggregation
df_daily = (
    df_fars_precip
    .groupby(["DATE", "STATE_ABBR"], as_index=False)["FATALS"]
    .sum()
    .rename(columns={"FATALS": "CRASH_COUNT"})
)
df_daily

,DATE,STATE_ABBR,CRASH_COUNT
0,1981-01-01,IN,1
1,1981-01-01,MD,3
2,1981-01-01,MI,1
3,1981-01-01,MN,4
4,1981-01-01,NC,1
...,...,...,...
91636,2019-12-31,IN,1
91637,2019-12-31,MA,2
91638,2019-12-31,MI,2
91639,2019-12-31,NY,1


# 2. Read in Wx Regimes & merge with FARS

In [10]:
# Load data
df_cluster = xr.open_dataset(cluster_file).to_dataframe().reset_index()

# Reset index so 'time' becomes a column
df_cluster = df_cluster.reset_index(drop=True)

# Rename the index column to 'DATE' to match df_fars
df_cluster = df_cluster.rename(columns={"time": "DATE"})

# Ensure datetime
df_cluster["DATE"] = pd.to_datetime(df_cluster["DATE"])

# Merge on DATE
df_merged = pd.merge(df_daily, df_cluster, on="DATE", how="left")

# Add cluster names
df_merged["cluster_name"] = df_merged["cluster"].map(cluster_names)

# Drop cluster number now that we have name
df_merged = df_merged.drop(columns='cluster')

df_merged

,DATE,STATE_ABBR,CRASH_COUNT,cluster_name
0,1981-01-01,IN,1,West Coast High
1,1981-01-01,MD,3,West Coast High
2,1981-01-01,MI,1,West Coast High
3,1981-01-01,MN,4,West Coast High
4,1981-01-01,NC,1,West Coast High
...,...,...,...,...
91636,2019-12-31,IN,1,Greenland High
91637,2019-12-31,MA,2,Greenland High
91638,2019-12-31,MI,2,Greenland High
91639,2019-12-31,NY,1,Greenland High


# 3. Get State precip data by weather regime and date

In [11]:
# Read in state precip data
df_state_precip=pd.read_csv(precip_file)

# Ensure Date time
df_state_precip["DATE"] = pd.to_datetime(df_state_precip["DATE"])

# Remove excluded states
df_filtered = df_state_precip[~df_state_precip["STATE_ABBR"].isin(excluded_states)]
df_filtered = df_filtered.drop(columns=["STATE", "STATEFP"])

# Merge cluster assignment into filtered state anomalies
df_final = df_filtered.merge(
    df_cluster,
    left_on='DATE',
    right_on='DATE',
    how='left'
)

# Add cluster names to df_final
df_final['cluster_name'] = df_final['cluster'].map(cluster_names)

# Drop cluster number now that we have name
df_final = df_final.drop(columns='cluster')

df_final

,DATE,ANOM_MEAN,ANOM_SUM,STATE_ABBR,cluster_name
0,1981-01-01,-4.391150,-917.750447,AL,West Coast High
1,1981-01-02,-4.417280,-923.211495,AL,West Coast High
2,1981-01-03,-4.426943,-925.231093,AL,West Coast High
3,1981-01-04,-4.419716,-923.720710,AL,West Coast High
4,1981-01-05,-4.411256,-921.952587,AL,West Coast High
...,...,...,...,...,...
168907,2019-12-27,-0.507263,-227.253931,WY,Pacific Trough
168908,2019-12-28,-0.266899,-119.570896,WY,Greenland High
168909,2019-12-29,-0.526915,-236.057830,WY,Greenland High
168910,2019-12-30,-0.526406,-235.829965,WY,Greenland High


In [12]:
# Merge daily crash counts into df_final
df_final = df_final.merge(
    df_daily,
    on=['DATE', 'STATE_ABBR'],
    how='left'  # keep all rows in df_final
)
df_final

,DATE,ANOM_MEAN,ANOM_SUM,STATE_ABBR,cluster_name,CRASH_COUNT
0,1981-01-01,-4.391150,-917.750447,AL,West Coast High,NaN
1,1981-01-02,-4.417280,-923.211495,AL,West Coast High,NaN
2,1981-01-03,-4.426943,-925.231093,AL,West Coast High,NaN
3,1981-01-04,-4.419716,-923.720710,AL,West Coast High,NaN
4,1981-01-05,-4.411256,-921.952587,AL,West Coast High,NaN
...,...,...,...,...,...,...
168907,2019-12-27,-0.507263,-227.253931,WY,Pacific Trough,NaN
168908,2019-12-28,-0.266899,-119.570896,WY,Greenland High,NaN
168909,2019-12-29,-0.526915,-236.057830,WY,Greenland High,NaN
168910,2019-12-30,-0.526406,-235.829965,WY,Greenland High,NaN


In [13]:
df_final['doy'] = df_final['DATE'].dt.dayofyear
climatology = (
    df_final
    .groupby(['STATE_ABBR', 'doy'], as_index=False)['CRASH_COUNT']
    .mean()
    .rename(columns={'CRASH_COUNT': 'CRASH_CLIM'})
)

df_final = df_final.merge(
    climatology,
    on=['STATE_ABBR', 'doy'],
    how='left'
)

In [14]:
# Daily crash anomaly
df_final['CRASH_COUNT'] = df_final['CRASH_COUNT'].fillna(0)
df_final['CRASH_ANOM'] = df_final['CRASH_COUNT'] - df_final['CRASH_CLIM']

df_final

,DATE,ANOM_MEAN,ANOM_SUM,STATE_ABBR,cluster_name,CRASH_COUNT,doy,CRASH_CLIM,CRASH_ANOM
0,1981-01-01,-4.391150,-917.750447,AL,West Coast High,0.0,1,1.833333,-1.833333
1,1981-01-02,-4.417280,-923.211495,AL,West Coast High,0.0,2,1.428571,-1.428571
2,1981-01-03,-4.426943,-925.231093,AL,West Coast High,0.0,3,1.200000,-1.200000
3,1981-01-04,-4.419716,-923.720710,AL,West Coast High,0.0,4,1.500000,-1.500000
4,1981-01-05,-4.411256,-921.952587,AL,West Coast High,0.0,5,2.444444,-2.444444
...,...,...,...,...,...,...,...,...,...
168907,2019-12-27,-0.507263,-227.253931,WY,Pacific Trough,0.0,361,1.000000,-1.000000
168908,2019-12-28,-0.266899,-119.570896,WY,Greenland High,0.0,362,1.000000,-1.000000
168909,2019-12-29,-0.526915,-236.057830,WY,Greenland High,0.0,363,NaN,NaN
168910,2019-12-30,-0.526406,-235.829965,WY,Greenland High,0.0,364,1.000000,-1.000000


In [15]:
# Count nonzero crash anomalies
num_nonzero = (df_final['CRASH_ANOM'] != 0).sum()
print(f"Number of nonzero crash anomalies: {num_nonzero}")

Number of nonzero crash anomalies: 166016


In [16]:
outFile='../data/combined_databases/database_fars_wxregimes_precip_daily.csv'
df_final.to_csv(outFile)

# 4. Make State Wxregime precip composites

In [17]:
# Compute state × cluster composites including names
df_composite = (
    df_final
    .groupby(['STATE_ABBR', 'cluster_name'], as_index=False)
    .agg({'ANOM_MEAN': 'mean'})
)

df_composite

,STATE_ABBR,cluster_name,ANOM_MEAN
0,AL,Alaskan Ridge/Pacific Ridge,1.161320
1,AL,Greenland High,0.253434
2,AL,Pacific Trough,-0.160282
3,AL,West Coast High,-1.307853
4,AR,Alaskan Ridge/Pacific Ridge,1.639292
...,...,...,...
187,WV,West Coast High,-1.099275
188,WY,Alaskan Ridge/Pacific Ridge,0.083645
189,WY,Greenland High,-0.057930
190,WY,Pacific Trough,0.071458


In [18]:
# Compute state × cluster composites including names
df_composite_crash = (
    df_final
    .groupby(['STATE_ABBR', 'cluster_name'], as_index=False)
    .agg({'CRASH_COUNT': 'sum'})
)

df_composite_crash

,STATE_ABBR,cluster_name,CRASH_COUNT
0,AL,Alaskan Ridge/Pacific Ridge,514.0
1,AL,Greenland High,246.0
2,AL,Pacific Trough,359.0
3,AL,West Coast High,272.0
4,AR,Alaskan Ridge/Pacific Ridge,338.0
...,...,...,...
187,WV,West Coast High,219.0
188,WY,Alaskan Ridge/Pacific Ridge,58.0
189,WY,Greenland High,40.0
190,WY,Pacific Trough,56.0


outFile='../data/combined_databases/database_fars_wxregimes_precip_daily.csv'
df_final.to_csv(outFile)

outFile='../data/combined_databases/database_fars_wxregimes_precip_daily_summary_state.csv'
df_composite.to_csv(outFile)